In [5]:
import pandas as pd
import json

In [6]:
df = pd.read_csv('../data/Milestone2Day1_contract_evaluation_dataset_sumedh_chandanshive.csv')
df.head()

,contract_id,contract_text,expected_apr,expected_term,expected_payment,expected_penalty
0,1,"This agreement includes APR 9.5%, tenure of 36...",9.5,36.0,450,NaN
1,2,The loan has a term of 48 months with a monthl...,NaN,48.0,620,NaN
2,3,APR is fixed at 11.2% with monthly payment $78...,11.2,NaN,780,Late fee $50
3,4,Customer agrees to a 24 month contract with AP...,6.8,24.0,NaN,NaN
4,5,Monthly installment shall be $510 for a period...,NaN,60.0,510,NaN


In [7]:
expected_output_format = {
    "apr": None,
    "term_months": None,
    "monthly_payment": None,
    "penalty_clause": None
}

In [8]:
PROMPT = """
You are an information extraction assistant.

Extract ONLY the following fields from the contract text:
- APR
- Term (months)
- Monthly payment
- Penalty clause

Rules:
- Output must be valid JSON only
- If a field is missing, return null
- Do not guess or infer
- Do not add extra keys

Return exactly this JSON structure:
{{
  "apr": null,
  "term_months": null,
  "monthly_payment": null,
  "penalty_clause": null
}}

Contract Text:
\"\"\"{text}\"\"\"
"""

In [9]:
def call_llm(prompt_text):
    """
    Dummy LLM response for baseline testing.
    """
    return json.dumps({
        "apr": None,
        "term_months": None,
        "monthly_payment": None,
        "penalty_clause": None
    })

In [10]:
def extract_fields(contract_text):
    prompt_filled = PROMPT.format(text=contract_text)
    llm_response = call_llm(prompt_filled)
    return json.loads(llm_response)

In [11]:
sample_contracts = df.sample(5, random_state=1)
sample_contracts

,contract_id,contract_text,expected_apr,expected_term,expected_payment,expected_penalty
14,15,Monthly payment $315 with late fee $30 if dela...,NaN,NaN,315,Late fee $30
13,14,This contract has a tenure of 72 months and AP...,9.75,72.0,NaN,NaN
17,18,APR 5.9% with no penalties mentioned.,5.90,NaN,NaN,NaN
3,4,Customer agrees to a 24 month contract with AP...,6.80,24.0,NaN,NaN
21,22,APR 14.6% applies with early termination charg...,14.60,NaN,NaN,Early termination fee $500


In [12]:
extracted_rows = []

for _, row in sample_contracts.iterrows():
    extracted = extract_fields(row["contract_text"])

    extracted_rows.append({
        "contract_text": row["contract_text"],

        "llm_apr": extracted["apr"],
        "llm_term": extracted["term_months"],
        "llm_payment": extracted["monthly_payment"],
        "llm_penalty": extracted["penalty_clause"],

        "expected_apr": row["expected_apr"],
        "expected_term": row["expected_term"],
        "expected_payment": row["expected_payment"],
        "expected_penalty": row["expected_penalty"]
    })

comparison_df = pd.DataFrame(extracted_rows)
comparison_df

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty
0,Monthly payment $315 with late fee $30 if dela...,None,None,None,None,NaN,NaN,315,Late fee $30
1,This contract has a tenure of 72 months and AP...,None,None,None,None,9.75,72.0,NaN,NaN
2,APR 5.9% with no penalties mentioned.,None,None,None,None,5.90,NaN,NaN,NaN
3,Customer agrees to a 24 month contract with AP...,None,None,None,None,6.80,24.0,NaN,NaN
4,APR 14.6% applies with early termination charg...,None,None,None,None,14.60,NaN,NaN,Early termination fee $500


In [13]:
def check_match(predicted, actual):
    if pd.isna(predicted) and pd.isna(actual):
        return 1
    if predicted == actual:
        return 1
    return 0

In [14]:
comparison_df["apr_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_apr"], r["expected_apr"]), axis=1
)

comparison_df["term_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_term"], r["expected_term"]), axis=1
)

comparison_df["payment_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_payment"], r["expected_payment"]), axis=1
)

comparison_df["penalty_match"] = comparison_df.apply(
    lambda r: check_match(r["llm_penalty"], r["expected_penalty"]), axis=1
)

comparison_df

,contract_text,llm_apr,llm_term,llm_payment,llm_penalty,expected_apr,expected_term,expected_payment,expected_penalty,apr_match,term_match,payment_match,penalty_match
0,Monthly payment $315 with late fee $30 if dela...,None,None,None,None,NaN,NaN,315,Late fee $30,1,1,0,0
1,This contract has a tenure of 72 months and AP...,None,None,None,None,9.75,72.0,NaN,NaN,0,0,1,1
2,APR 5.9% with no penalties mentioned.,None,None,None,None,5.90,NaN,NaN,NaN,0,1,1,1
3,Customer agrees to a 24 month contract with AP...,None,None,None,None,6.80,24.0,NaN,NaN,0,0,1,1
4,APR 14.6% applies with early termination charg...,None,None,None,None,14.60,NaN,NaN,Early termination fee $500,0,1,1,0
